# Phase 3 — Data Cleaning

## Goal
Take the raw dataset, apply the cleaning decisions documented in `src/data/clean_data.py`, and save a single canonical clean version to `data/processed/`. From this point on, every downstream notebook loads the **clean** file, not the raw one.

## Why a module instead of notebook-only code
Notebook code is exploration. Production-grade ML re-uses the same cleaning logic at training and at inference. If we duplicate the cleaning rules across notebooks, training and serving will drift apart silently — the classic 'works in my notebook' bug. By writing it once in `clean_data.py`, both the modeling notebook *and* the Streamlit app call the same function.

## What we're applying
1. Drop `customerID` (identifier, no signal)
2. Coerce `TotalCharges` to numeric, impute the 11 tenure-0 blanks with 0
3. Encode `Churn` target as int (Yes→1, No→0)
4. Normalize `SeniorCitizen` to Yes/No strings
5. Strip whitespace defensively

Every line is defensible in interview — see the docstring in `clean_data.py` for the reasoning.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
pd.set_option("display.max_columns", None)

from src.data.load_data import load_raw_telco
from src.data.clean_data import clean_telco, save_processed, get_feature_columns

# Load raw and clean
raw = load_raw_telco()
print(f"Raw shape: {raw.shape}")
print(f"Raw TotalCharges dtype: {raw['TotalCharges'].dtype}")
print(f"Raw Churn unique values: {raw['Churn'].unique()}")

Raw shape: (7043, 21)
Raw TotalCharges dtype: object
Raw Churn unique values: ['No' 'Yes']


In [2]:
clean = clean_telco(raw)
print(f"Clean shape: {clean.shape}")
print(f"\nClean dtypes:\n{clean.dtypes}")
print(f"\nChurn distribution after encoding:\n{clean['Churn'].value_counts(normalize=True).round(4)}")
print(f"\nTotalCharges summary:\n{clean['TotalCharges'].describe().round(2)}")
print(f"\nAny remaining nulls? {clean.isna().sum().sum()}")

Clean shape: (7043, 20)

Clean dtypes:
gender               object
SeniorCitizen        object
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges        float64
Churn                 int32
dtype: object

Churn distribution after encoding:
Churn
0    0.7346
1    0.2654
Name: proportion, dtype: float64

TotalCharges summary:
count    7043.00
mean     2279.73
std      2266.79
min         0.00
25%       398.55
50%      1394.55
75%      3786.60
max      8684.80
Name: TotalCharges, dtype: float64

Any remaining nulls? 0


In [3]:
# Verify the 11 tenure-0 rows are now imputed with 0
zero_tenure_check = clean.loc[clean["tenure"] == 0, ["tenure", "MonthlyCharges", "TotalCharges", "Churn"]]
print(f"Rows with tenure=0 (should all have TotalCharges=0):\n")
zero_tenure_check

Rows with tenure=0 (should all have TotalCharges=0):



,tenure,MonthlyCharges,TotalCharges,Churn
488,0,52.55,0.0,0
753,0,20.25,0.0,0
936,0,80.85,0.0,0
1082,0,25.75,0.0,0
1340,0,56.05,0.0,0
3331,0,19.85,0.0,0
3826,0,25.35,0.0,0
4380,0,20.00,0.0,0
5218,0,19.70,0.0,0
6670,0,73.35,0.0,0


In [4]:
# Inspect feature column groupings — these define what goes into the model later
features = get_feature_columns()
for group, cols in features.items():
    print(f"{group}: {cols}")

numeric: ['tenure', 'MonthlyCharges', 'TotalCharges']
binary_categorical: ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']
multi_categorical: ['MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaymentMethod']
target: Churn


In [5]:
# Save the cleaned dataset — this is the canonical version every downstream step uses
out_path = save_processed(clean, filename="telco_clean.csv")
print(f"Saved cleaned dataset to: {out_path}")
print(f"File size: {out_path.stat().st_size / 1024:.1f} KB")

Saved cleaned dataset to: C:\Users\egorimanikon\OneDrive - Stony Brook University\Desktop\Churn Project\Churn Project\data\processed\telco_clean.csv
File size: 879.6 KB


## Phase 3 summary

**Interview line** — *"In cleaning, I made five decisions, each defended in the module docstring: dropped customerID as a non-signal identifier, coerced TotalCharges to numeric and imputed the 11 tenure-zero rows with 0, encoded Churn to 0/1, normalized SeniorCitizen for downstream consistency, and stripped whitespace defensively. I kept cleaning in a Python module rather than the notebook so the training pipeline and Streamlit inference call the same function — no train/serve skew."*